In [1]:
from google import genai
import os 
import pandas as pd

# The client gets the API key from the environment variable `GEMINI_API_KEY`.
client = genai.Client()
api_key = os.getenv('GEMINI_API_KEY')
if api_key:
    print(f"GEMINI_API_KEY was found.")
else:
    print("GEMINI_API_KEY not found in environment variables.")

GEMINI_API_KEY was found.


In [ ]:
from google import genai
from pydantic import BaseModel, Field
from typing import List, Optional
import os
client = genai.Client()

link = "https://www.sec.gov/ix?doc=/Archives/edgar/data/0001257640/000110465926025219/kro-20251231x10k.htm"
prompt = f"""
Look at this link, show me revenue segments and its mix, and its top competitors in each segment. 
{link}
"""


class _Revenue(BaseModel):
    Segment: str = Field(description="Name of the operating segment.")
    Fiscal_year: str = Field(description="The fiscal year for which the revenue is reported. and the corresponding calendar period.")
    Revenue_dollars: str = Field(description="dollar amount of the segment's revenue.")
    Revenue_share: str = Field(description="The percentage of total revenue that this segment contributes.")
    Revenue_growth: Optional[str] = Field(description="The growth rate of the segment's revenue.")
    miscellaneous_info: Optional[str] = Field(description="Any additional information about the segment.")

class _Competitors(BaseModel):
    name: str = Field(description="Name of the competitor, and its public ticker if available.")
    is_direct_competitor: Optional[str] = Field(description="Yes if it's direct competitor otherwise No.")
    competiting_segment: Optional[str] = Field(description="The segment in which the competitor operates.")
    miscellaneous_info: Optional[str] = Field(description="Any additional information about the competitor.")
    
#
class BUSINESS(BaseModel):
    company_name: str = Field(description="The name of the company.")
    revenue: list[_Revenue]
    competitors: list[_Competitors] 

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=prompt,
    config={
        "response_mime_type": "application/json",
        "response_json_schema": BUSINESS.model_json_schema(),
    },
)
revenue = BUSINESS.model_validate_json(response.text)



In [ ]:
Please review the credit agreement at the SEC link below and answer the following questions using only the agreement as the source.

https://www.sec.gov/Archives/edgar/data/1041859/000104185925000015/plce-ex106x11125.htm

Instructions:
- Use only the linked agreement.
- Be precise and conservative.
- Do not infer provisions unless clearly supported by the agreement.
- If an item is not clearly stated, say: "Not clearly stated in the provided agreement."
- Cite relevant section numbers, articles, schedules, or exhibits for each answer.
- Answer each question separately and in order.

Questions:
1. Who are the key lenders and what're their roles and commitment allocations (administrative agent, Lead Arranger and/or bookrunners)?
2. What are the facilities, including size, maturity, and any sublimits?
3. Is there any accordion or incremental feature?
4. What are the interest rate options, applicable margins, pricing grid mechanics, and fees?
5. Is there any borrowing base or collateral-based availability concept?
6. Is there any amortization schedule for term loans?
7. What are the financial covenants, including thresholds, testing frequency, and any springing feature?
8. What are the voluntary and mandatory prepayment provisions?
9. What are the key restrictions on acquisitions, investments, asset sales, and dispositions?
10. What defined terms are most important for covenant analysis, especially EBITDA and leverage-related definitions?

For each item, provide:
- Direct answer
- Key details
- Section references
- Caveats